# Chest X-ray report generation — ResNet + GPT-2 (Kaggle, T4)

**Run it with "Run All".** No `pip install` is needed. Turn **Internet ON** (right panel → Session options) so GPT-2 and the ImageNet ResNet weights can be downloaded.

How it works: frontal + lateral X-ray → ImageNet-pretrained ResNet → 2 × 49 visual tokens → used as a *prefix* for GPT-2 (all GPT-2 weights are fine-tuned) → the report is generated with beam search.
The notebook runs a short **pre-flight check** before the long training, so a mistake shows up in about a minute instead of after hours.

In [ ]:
# 1) Setup + config (no pip install: everything used here ships with Kaggle's image)
import os, sys, re, json, math, time, glob, gc, random, hashlib, textwrap
from collections import Counter
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.dataloader import default_collate
import torchvision, transformers
from torchvision import models as tvm
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import GroupShuffleSplit
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True
import warnings
warnings.filterwarnings("ignore", message=".*no_repeat_ngram_size.*")   # harmless generate() notice

# ===================== CONFIG: the only place you may need to edit =====================
CSV_NAME        = "chest_xray_frontal_lateral_reports.csv"
GPT2_SOURCE     = "gpt2"      # Hugging Face id (needs Internet ON) or a local folder with GPT-2 files
ENCODER         = "resnet50"  # resnet18 | resnet34 | resnet50  (ImageNet-pretrained image encoder)
IMG_SIZE        = 224
BATCH_SIZE      = 16          # halved automatically if the GPU runs out of memory
EPOCHS          = 8
PATIENCE        = 2           # stop when validation loss has not improved for this many epochs
LR_ENC, LR_HEAD, LR_LM = 1e-4, 5e-4, 5e-5
FREEZE_LM_STEPS = 200         # first train only the image->text bridge, then unfreeze GPT-2
WARMUP_STEPS    = 100
MAX_TRAIN       = None        # e.g. 800 for a quick dry run
N_EVAL          = 500         # number of test studies to generate reports for (None = all)
NUM_BEAMS       = 3
SEED            = 42
OUT_DIR         = "/kaggle/working/xray_report"
# ======================================================================================

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = True
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| torchvision", torchvision.__version__, "| transformers", transformers.__version__)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if USE_AMP else "no GPU (this will be very slow)")

In [ ]:
# 2) Load + clean the CSV and make sure the image paths work
def find_file(name):
    hits = []
    for depth in range(1, 5):                       # shallow search only: never walks the huge image folders
        hits += glob.glob("/kaggle/input/" + "/".join(["*"] * (depth - 1) + [name]))
    return sorted(set(hits), key=lambda p: (len(p), p))

hits = find_file(CSV_NAME)
if not hits:
    raise FileNotFoundError(f"{CSV_NAME} not found under /kaggle/input. Click 'Add Input' and attach the dataset that contains it.")
CSV_PATH = hits[0]
print("Using CSV:", CSV_PATH)

df = pd.read_csv(CSV_PATH)
need = ["subject_id", "study_id", "frontal_path", "lateral_path", "text"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"CSV is missing columns {missing}. Available: {list(df.columns)}")
df = df.dropna(subset=need).copy()
df["text"] = df.text.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[(df.text != "") & (df.text.str.lower() != "nan")]
df = df.drop_duplicates("study_id").reset_index(drop=True)
for c in ("frontal_path", "lateral_path"):
    df[c] = df[c].astype(str).str.strip().str.replace("\\", "/", regex=False)

def frac_ok(col, n=200):
    s = df[col].sample(min(n, len(df)), random_state=0)
    return float(np.mean([os.path.isfile(p) for p in s]))

if min(frac_ok("frontal_path"), frac_ok("lateral_path")) < 0.99:
    # The CSV was made on another machine: find where the "files/" folder lives in this session.
    ex = df.frontal_path.iloc[0]
    suffix = ex.split("/files/", 1)[1] if "/files/" in ex else ex
    roots = sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*") + glob.glob("/kaggle/input/*/*/*"))
    base = None
    for r in roots:
        for rel in ("official_data_iccv_final/files", "files", ""):
            cand = os.path.join(r, rel)
            if os.path.isfile(os.path.join(cand, suffix)):
                base = cand; break
        if base: break
    if base is None:
        raise FileNotFoundError(f"Images not found (looked for '{suffix}'). Attach the image dataset with 'Add Input'.")
    for c in ("frontal_path", "lateral_path"):
        df[c] = df[c].map(lambda p: os.path.join(base, p.split("/files/", 1)[1] if "/files/" in p else p))
    print("Remapped image paths to:", base)
print(f"Studies: {len(df)} | patients: {df.subject_id.nunique()} | image paths OK: "
      f"{frac_ok('frontal_path'):.0%} frontal, {frac_ok('lateral_path'):.0%} lateral (sample of 200)")
print("Example report:", df.text.iloc[0][:250])

In [ ]:
# 3) Read every image ONCE, resize, and cache as a uint8 array (makes each training epoch minutes, not hours)
S = IMG_SIZE
def load_gray(path):
    try:
        with Image.open(path) as im:
            im.draft("L", (S * 2, S * 2))            # fast JPEG decode at reduced size
            return np.asarray(im.convert("L").resize((S, S), Image.BILINEAR), dtype=np.uint8)
    except Exception:
        return None

N = len(df)
key = hashlib.md5(("|".join(df.frontal_path.tolist() + df.lateral_path.tolist()) + str(S)).encode()).hexdigest()[:10]
cache_img, cache_ok = f"{OUT_DIR}/imgcache_{S}_{key}.npy", f"{OUT_DIR}/imgcache_{S}_{key}_ok.npy"

if os.path.exists(cache_img) and os.path.exists(cache_ok):
    imgs, valid = np.load(cache_img), np.load(cache_ok)
    print("Loaded image cache:", imgs.shape)
else:
    mm = np.lib.format.open_memmap(cache_img, mode="w+", dtype=np.uint8, shape=(N, 2, S, S))
    valid = np.zeros(N, dtype=bool)
    fp, lp = df.frontal_path.tolist(), df.lateral_path.tolist()
    def work(i):
        a, b = load_gray(fp[i]), load_gray(lp[i])
        if a is None or b is None: return False
        mm[i, 0], mm[i, 1] = a, b
        return True
    workers = min(16, (os.cpu_count() or 2) * 4)
    with ThreadPoolExecutor(workers) as ex:
        for i, ok in enumerate(tqdm(ex.map(work, range(N)), total=N, desc="caching images")):
            valid[i] = ok
    mm.flush(); del mm
    np.save(cache_ok, valid)
    imgs = np.load(cache_img)

n_bad = int((~valid).sum())
print(f"Unreadable studies: {n_bad} / {N}")
if n_bad > 0.2 * N:
    raise RuntimeError("More than 20% of the images could not be read: the image paths are wrong.")
if n_bad:
    df, imgs = df[valid].reset_index(drop=True), imgs[valid]
assert imgs[:64].mean() > 3, "Images look empty (all black): check the image paths"
print("Image array:", imgs.shape, imgs.dtype, f"({imgs.nbytes / 2**30:.1f} GB in RAM) | mean pixel {imgs[:64].mean():.1f}")

In [ ]:
# 4) Tokenizer: the ORIGINAL GPT-2 vocabulary, no new tokens (so no vocab mismatch is possible)
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
try:
    tokenizer = GPT2TokenizerFast.from_pretrained(GPT2_SOURCE)
except Exception as e:
    raise RuntimeError("Could not load the GPT-2 tokenizer. Turn Internet ON (Session options) or attach a GPT-2 "
                       "dataset and set GPT2_SOURCE to its folder.") from e
tokenizer.pad_token = tokenizer.eos_token
BOS_ID = EOS_ID = PAD_ID = tokenizer.eos_token_id          # GPT-2 uses <|endoftext|> for all three
sample = df.text.iloc[0]
enc0 = tokenizer(sample, add_special_tokens=False)["input_ids"]
assert len(enc0) > 5 and tokenizer.decode(enc0)[:15].lower() == sample[:15].lower(), "Tokenizer is broken"
print("Tokenizer vocab:", len(tokenizer), "| eos/bos/pad id:", EOS_ID, "| sample tokens:", len(enc0))

enc = tokenizer(df.text.tolist(), add_special_tokens=False)["input_ids"]
lens = np.array([len(t) for t in enc])
p95 = int(np.percentile(lens, 95))
MAX_LEN = int(min(192, max(48, math.ceil((p95 + 2) / 8) * 8)))   # sequence length incl. BOS/EOS
print(f"Report length in tokens: median {int(np.median(lens))}, p95 {p95}, max {lens.max()} -> MAX_LEN = {MAX_LEN} "
      f"({(lens > MAX_LEN - 2).mean():.1%} of reports get truncated)")

ids_arr  = np.full((len(df), MAX_LEN), PAD_ID, dtype=np.int64)
mask_arr = np.zeros((len(df), MAX_LEN), dtype=np.int64)
for i, t in enumerate(enc):
    seq = [BOS_ID] + t + [EOS_ID] if len(t) <= MAX_LEN - 2 else [BOS_ID] + t[:MAX_LEN - 1]
    ids_arr[i, :len(seq)] = seq
    mask_arr[i, :len(seq)] = 1
assert ids_arr.max() < len(tokenizer)
print("First training sequence:", tokenizer.decode(ids_arr[0][mask_arr[0] == 1])[:200])

In [ ]:
# 5) Patient-level split (no patient appears in two sets) + data loaders
trv_idx, te_idx = next(GroupShuffleSplit(1, test_size=0.05, random_state=SEED).split(df, groups=df.subject_id))
tr_rel, va_rel = next(GroupShuffleSplit(1, test_size=0.05 / 0.95, random_state=SEED).split(trv_idx, groups=df.subject_id.values[trv_idx]))
tr_idx, va_idx = trv_idx[tr_rel], trv_idx[va_rel]
if MAX_TRAIN: tr_idx = tr_idx[:MAX_TRAIN]
sets = {k: set(df.subject_id.values[v]) for k, v in dict(train=tr_idx, val=va_idx, test=te_idx).items()}
assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"]), "patient leakage"
print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(te_idx)} studies (patient-disjoint)")

class XrayDS(Dataset):
    def __init__(self, idx): self.idx = np.asarray(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        return torch.from_numpy(imgs[i]), torch.from_numpy(ids_arr[i]), torch.from_numpy(mask_arr[i])

train_ds, val_ds, test_ds = XrayDS(tr_idx), XrayDS(va_idx), XrayDS(te_idx)
def make_loaders(bs):
    kw = dict(num_workers=0, pin_memory=USE_AMP)
    return (DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True, **kw),
            DataLoader(val_ds, batch_size=bs * 2, shuffle=False, **kw))
train_loader, val_loader = make_loaders(BATCH_SIZE)
print("train batches per epoch:", len(train_loader))

In [ ]:
# 6) The model
IMG_MEAN, IMG_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def augment(x):
    # x: (N,1,H,W) in [0,1]. Mild rotation / zoom / shift + brightness / contrast. No flips (left/right matters).
    n, dev = x.size(0), x.device
    ang   = (torch.rand(n, device=dev) - 0.5) * 2 * math.radians(5)
    scale = 1 + (torch.rand(n, device=dev) - 0.5) * 0.2
    tx, ty = [(torch.rand(n, device=dev) - 0.5) * 0.1 for _ in range(2)]
    cos, sin = torch.cos(ang) / scale, torch.sin(ang) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, list(x.shape), align_corners=False),
                      mode="bilinear", padding_mode="border", align_corners=False)
    b = (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.2
    c = 1 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 0.2
    return ((x - 0.5) * c + 0.5 + b).clamp(0, 1)

def make_encoder(name):
    weights = {"resnet18": tvm.ResNet18_Weights, "resnet34": tvm.ResNet34_Weights, "resnet50": tvm.ResNet50_Weights}[name]
    try:
        return getattr(tvm, name)(weights=weights.DEFAULT), True
    except Exception as e:
        print(f"WARNING: could not download ImageNet weights ({type(e).__name__}). Using a randomly initialised "
              f"{name}: results will be clearly worse. Turn Internet ON and re-run.")
        return getattr(tvm, name)(weights=None), False

class XrayReportModel(nn.Module):
    def __init__(self, lm, encoder_name, grid, p_drop=0.1):
        super().__init__()
        self.lm, self.grid, self.n_vis = lm, grid, 2 * grid * grid
        E = lm.config.n_embd
        net, self.pretrained = make_encoder(encoder_name)
        self.encoder = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1, net.layer2, net.layer3, net.layer4)
        self.proj = nn.Sequential(nn.Linear(net.fc.in_features, E), nn.GELU(), nn.Linear(E, E))
        self.pos  = nn.Parameter(torch.randn(1, grid * grid, E) * 0.02)     # where in the image
        self.view = nn.Parameter(torch.randn(1, 2, 1, E) * 0.02)            # frontal vs lateral
        self.norm = nn.LayerNorm(E)
        nn.init.constant_(self.norm.weight, 0.15)                           # start at the scale of GPT-2 word embeddings
        self.drop = nn.Dropout(p_drop)
        self.register_buffer("mean", torch.tensor(IMG_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std",  torch.tensor(IMG_STD).view(1, 3, 1, 1), persistent=False)
        self._warned = False

    def encode_images(self, x_u8):
        B, _, H, W = x_u8.shape
        x = x_u8.reshape(B * 2, 1, H, W).float().div(255.0)
        if self.training: x = augment(x)
        x = (x.expand(-1, 3, -1, -1) - self.mean) / self.std
        f = F.adaptive_avg_pool2d(self.encoder(x), self.grid).flatten(2).transpose(1, 2)   # (2B, g*g, C)
        t = (self.proj(f) + self.pos).view(B, 2, self.grid * self.grid, -1) + self.view
        return self.drop(self.norm(t.reshape(B, self.n_vis, -1)))                          # (B, 2*g*g, E)

    def forward(self, imgs, ids, mask):
        # Returns (sum of token losses, number of tokens). Position t predicts token t+1; the loss is computed here
        # explicitly, so there is no hidden label shifting.
        mask = mask.long()
        prefix = self.encode_images(imgs)
        tok = self.lm.transformer.wte(ids)
        embeds = torch.cat([prefix.to(tok.dtype), tok], 1)
        attn = torch.cat([torch.ones(ids.size(0), self.n_vis, dtype=mask.dtype, device=mask.device), mask], 1)
        h = self.lm.transformer(inputs_embeds=embeds, attention_mask=attn, use_cache=False).last_hidden_state[:, self.n_vis:]
        logits = self.lm.lm_head(h[:, :-1]).float()
        tgt = ids[:, 1:].masked_fill(mask[:, 1:] == 0, -100)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="sum")
        return loss, (tgt != -100).sum()

    @torch.no_grad()
    def generate(self, imgs, max_new_tokens, num_beams=3, no_repeat_ngram_size=5):
        self.eval()
        prefix = self.encode_images(imgs)
        bos = self.lm.transformer.wte(torch.full((imgs.size(0), 1), BOS_ID, device=imgs.device))
        embeds = torch.cat([prefix.to(bos.dtype), bos], 1)
        attn = torch.ones(embeds.shape[:2], dtype=torch.long, device=embeds.device)
        try:
            return self.lm.generate(inputs_embeds=embeds, attention_mask=attn, max_new_tokens=max_new_tokens,
                                    min_new_tokens=5, num_beams=num_beams, no_repeat_ngram_size=no_repeat_ngram_size,
                                    early_stopping=True, do_sample=False, eos_token_id=EOS_ID, pad_token_id=PAD_ID, use_cache=True)
        except Exception as e:                                   # safety net: plain greedy decoding
            if not self._warned:
                print("generate() failed, falling back to greedy decoding:", repr(e)[:200]); self._warned = True
            return self._greedy(embeds, attn, max_new_tokens)

    def _greedy(self, embeds, attn, max_new_tokens):
        out, done = [], torch.zeros(embeds.size(0), dtype=torch.bool, device=embeds.device)
        for _ in range(max_new_tokens):
            h = self.lm.transformer(inputs_embeds=embeds, attention_mask=attn, use_cache=False).last_hidden_state[:, -1]
            nxt = self.lm.lm_head(h).argmax(-1).masked_fill(done, PAD_ID)
            out.append(nxt); done |= nxt == EOS_ID
            if done.all(): break
            embeds = torch.cat([embeds, self.lm.transformer.wte(nxt[:, None]).to(embeds.dtype)], 1)
            attn = torch.cat([attn, torch.ones_like(attn[:, :1])], 1)
        return torch.stack(out, 1)

def decode_generated(out):
    texts = []
    for row in out.tolist():
        if EOS_ID in row: row = row[:row.index(EOS_ID)]
        texts.append(tokenizer.decode(row, skip_special_tokens=True).strip())
    return texts

def decode_refs(ids, mask):
    texts = []
    for r, m in zip(ids.tolist(), mask.tolist()):
        r = r[1:sum(m)]                                          # drop BOS and padding
        if r and r[-1] == EOS_ID: r = r[:-1]
        texts.append(tokenizer.decode(r, skip_special_tokens=True).strip())
    return texts

def build_model():
    lm = GPT2LMHeadModel.from_pretrained(GPT2_SOURCE).float()
    assert lm.get_input_embeddings().num_embeddings >= len(tokenizer), "GPT-2 vocabulary smaller than the tokenizer"
    grid = max(1, IMG_SIZE // 32)
    assert lm.config.n_positions >= 2 * grid * grid + MAX_LEN, "GPT-2 context window too small: lower IMG_SIZE or MAX_LEN"
    return XrayReportModel(lm, ENCODER, grid).to(DEVICE)

def set_lm_trainable(model, flag):
    for p in model.lm.parameters(): p.requires_grad = flag

def make_optimizer(model):
    head = list(model.proj.parameters()) + [model.pos, model.view] + list(model.norm.parameters())
    groups = []
    for params, lr in ((list(model.encoder.parameters()), LR_ENC), (head, LR_HEAD), (list(model.lm.parameters()), LR_LM)):
        groups.append(dict(params=[p for p in params if p.ndim >= 2], lr=lr, weight_decay=0.01))
        groups.append(dict(params=[p for p in params if p.ndim < 2], lr=lr, weight_decay=0.0))
    return torch.optim.AdamW([g for g in groups if g["params"]])

def lr_factor(step, total):
    warm = min(1.0, (step + 1) / max(1, WARMUP_STEPS))
    cos = 0.5 * (1 + math.cos(math.pi * min(1.0, step / max(1, total))))
    return warm * max(0.02, cos)

@torch.no_grad()
def evaluate_loss(model, loader, shuffle_images=False):
    model.eval(); tot, cnt = 0.0, 0
    for imgs_b, ids_b, mask_b in loader:
        imgs_b, ids_b, mask_b = imgs_b.to(DEVICE), ids_b.to(DEVICE), mask_b.to(DEVICE)
        if shuffle_images: imgs_b = torch.roll(imgs_b, 1, 0)     # pair each report with another patient's X-ray
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            ls, n = model(imgs_b, ids_b, mask_b)
        tot += ls.item(); cnt += n.item()
    return tot / max(cnt, 1)

_m = build_model()
n_all = sum(p.numel() for p in _m.parameters())
print(f"Model built: {n_all/1e6:.0f}M parameters | encoder {ENCODER} (ImageNet-pretrained: {_m.pretrained}) | "
      f"{_m.n_vis} visual tokens + up to {MAX_LEN} text tokens per report")
del _m; gc.collect()

In [ ]:
# 7) PRE-FLIGHT: ~1 minute of checks on one batch before the long run (loss must be finite and fall, generation must work, memory must fit)
def preflight(bs, steps=25):
    m = build_model(); set_lm_trainable(m, True); m.train()
    opt = make_optimizer(m); scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    imgs_b, ids_b, mask_b = [t.to(DEVICE) for t in default_collate([train_ds[i] for i in range(bs)])]
    params = [p for p in m.parameters() if p.requires_grad]
    if USE_AMP: torch.cuda.reset_peak_memory_stats()
    losses = []
    for _ in range(steps):
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            ls, n = m(imgs_b, ids_b, mask_b); loss = ls / n
        assert torch.isfinite(loss), "non-finite loss in pre-flight"
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
        losses.append(loss.item())
    peak = torch.cuda.max_memory_allocated() / 2**30 if USE_AMP else 0.0
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        gen = decode_generated(m.generate(imgs_b[:2], max_new_tokens=12, num_beams=NUM_BEAMS))
    del m, opt, scaler, imgs_b, ids_b, mask_b, params; gc.collect()
    if USE_AMP: torch.cuda.empty_cache()
    return losses, gen, peak

while True:
    try:
        losses, gen, peak = preflight(BATCH_SIZE); break
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if BATCH_SIZE == 1: raise
        BATCH_SIZE //= 2; train_loader, val_loader = make_loaders(BATCH_SIZE)
        print(f"Out of GPU memory -> batch size reduced to {BATCH_SIZE}")

print(f"batch size {BATCH_SIZE} | loss {losses[0]:.2f} -> {losses[-1]:.2f} over {len(losses)} steps on one batch | peak GPU memory {peak:.1f} GB")
print("generation works, sample output:", gen[0][:100] if gen else gen)
if not losses[-1] < losses[0] - 0.1:
    print("WARNING: loss barely moved in the pre-flight. Training will still run, but check the data/paths if the real loss does not fall.")
else:
    print("PRE-FLIGHT OK")

In [ ]:
# 8) Train (best checkpoint by validation loss is kept; early stopping)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model = build_model()
set_lm_trainable(model, False)                                   # phase 1: only the image bridge learns
opt = make_optimizer(model)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
total_steps = EPOCHS * len(train_loader)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: lr_factor(s, total_steps))
BEST_PATH = f"{OUT_DIR}/best.pt"

history, step, best, bad = [], 0, float("inf"), 0
t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train(); run_sum = torch.zeros((), device=DEVICE); run_n = torch.zeros((), device=DEVICE, dtype=torch.long); skipped = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False)
    for imgs_b, ids_b, mask_b in pbar:
        if step == FREEZE_LM_STEPS: set_lm_trainable(model, True)  # phase 2: fine-tune GPT-2 as well
        imgs_b, ids_b, mask_b = imgs_b.to(DEVICE, non_blocking=True), ids_b.to(DEVICE, non_blocking=True), mask_b.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            ls, n = model(imgs_b, ids_b, mask_b); loss = ls / n.clamp(min=1)
        if not torch.isfinite(loss):
            skipped += 1; opt.zero_grad(set_to_none=True); sched.step(); step += 1
            if skipped > 20: raise RuntimeError("Too many non-finite losses: training is unstable")
            continue
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(opt); scaler.update(); sched.step(); step += 1
        run_sum += ls.detach(); run_n += n
        if step % 25 == 0: pbar.set_postfix(loss=f"{(run_sum / run_n).item():.3f}")
    train_loss = (run_sum / run_n.clamp(min=1)).item()
    val_loss = evaluate_loss(model, val_loader)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss, val_ppl=math.exp(val_loss)))
    note = ""
    if val_loss < best - 1e-4:
        best, bad = val_loss, 0; torch.save(model.state_dict(), BEST_PATH); note = "  <- best, saved"
    else:
        bad += 1
    print(f"epoch {epoch}/{EPOCHS} | train loss {train_loss:.3f} | val loss {val_loss:.3f} (ppl {math.exp(val_loss):.1f}) | "
          f"{(time.time() - t_start) / 60:.1f} min elapsed{note}")
    if bad >= PATIENCE:
        print("Early stopping: validation loss stopped improving."); break

pd.DataFrame(history).to_csv(f"{OUT_DIR}/training_history.csv", index=False)
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
json.dump(dict(ENCODER=ENCODER, IMG_SIZE=IMG_SIZE, MAX_LEN=MAX_LEN, GPT2_SOURCE=GPT2_SOURCE, best_val_loss=best), open(f"{OUT_DIR}/config.json", "w"))
print(f"Done. Best validation loss {best:.3f} (perplexity {math.exp(best):.1f}). Saved to {BEST_PATH}")

In [ ]:
# 9) Learning curves
h = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(h.epoch, h.train_loss, "o-", label="train"); ax.plot(h.epoch, h.val_loss, "o-", label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy per token"); ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/learning_curves.png", dpi=120); plt.show()
print(h.round(3).to_string(index=False))

In [ ]:
# 10) Evaluation on the held-out TEST patients: BLEU-1..4, ROUGE-1/2/L, and a check that the model really uses the image
def toks(s): return re.findall(r"\w+", s.lower())
def ngram_counts(t, n): return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))

def corpus_bleu(hyps, refs):
    clipped, total, hl, rl = [0] * 4, [0] * 4, 0, 0
    for h, r in zip(hyps, refs):
        hl += len(h); rl += len(r)
        for n in range(1, 5):
            hc, rc = ngram_counts(h, n), ngram_counts(r, n)
            clipped[n - 1] += sum(min(c, rc[g]) for g, c in hc.items()); total[n - 1] += max(len(h) - n + 1, 0)
    bp = 1.0 if hl > rl else (math.exp(1 - rl / hl) if hl else 0.0)
    out = {}
    for n in range(1, 5):
        ok = all(clipped[i] > 0 and total[i] > 0 for i in range(n))
        out[f"BLEU-{n}"] = bp * math.exp(sum(math.log(clipped[i] / total[i]) for i in range(n)) / n) if ok else 0.0
    return out

def f1(p, r): return 2 * p * r / (p + r) if p + r else 0.0
def rouge_n(h, r, n):
    hc, rc = ngram_counts(h, n), ngram_counts(r, n)
    ov = sum(min(c, rc[g]) for g, c in hc.items())
    return f1(ov / max(sum(hc.values()), 1), ov / max(sum(rc.values()), 1))
def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b): cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]
def rouge_l(h, r):
    l = lcs_len(h, r)
    return f1(l / max(len(h), 1), l / max(len(r), 1))

model.eval()
n_eval = len(test_ds) if not N_EVAL else min(N_EVAL, len(test_ds))
eval_loader = DataLoader(torch.utils.data.Subset(test_ds, range(n_eval)), batch_size=32, shuffle=False)
hyps, refs = [], []
for imgs_b, ids_b, mask_b in tqdm(eval_loader, desc="generating reports"):
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        out = model.generate(imgs_b.to(DEVICE), max_new_tokens=MAX_LEN - 1, num_beams=NUM_BEAMS)
    hyps += decode_generated(out); refs += decode_refs(ids_b, mask_b)

H, R = [toks(x) for x in hyps], [toks(x) for x in refs]
scores = corpus_bleu(H, R)
scores["ROUGE-1"] = float(np.mean([rouge_n(h, r, 1) for h, r in zip(H, R)]))
scores["ROUGE-2"] = float(np.mean([rouge_n(h, r, 2) for h, r in zip(H, R)]))
scores["ROUGE-L"] = float(np.mean([rouge_l(h, r) for h, r in zip(H, R)]))
scores["avg_words_generated"] = float(np.mean([len(h) for h in H]))
scores["avg_words_reference"] = float(np.mean([len(r) for r in R]))
scores["distinct_reports_%"]  = 100 * len(set(hyps)) / len(hyps)

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
loss_true, loss_shuf = evaluate_loss(model, test_loader), evaluate_loss(model, test_loader, shuffle_images=True)
scores["test_loss_with_correct_image"], scores["test_loss_with_wrong_image"] = loss_true, loss_shuf

print(f"=== Test results on {len(hyps)} studies ===")
for k, v in scores.items(): print(f"{k:32s}{v:9.4f}")
print(f"\nImage check: loss with the right X-ray {loss_true:.3f} vs a wrong patient's X-ray {loss_shuf:.3f} -> "
      + ("the model uses the images." if loss_shuf > loss_true + 0.01 else "the model barely uses the images (it behaves like a text-only model)."))

pd.DataFrame(dict(study_id=df.study_id.values[te_idx[:len(hyps)]], reference=refs, generated=hyps)).to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
json.dump(scores, open(f"{OUT_DIR}/test_metrics.json", "w"), indent=2)

In [ ]:
# 11) A few examples (X-ray, real report, generated report)
k = min(3, len(hyps))
fig, axes = plt.subplots(k, 2, figsize=(13, 3.2 * k), gridspec_kw=dict(width_ratios=[1, 3.2]))
axes = np.atleast_2d(axes)
for r in range(k):
    axes[r, 0].imshow(imgs[te_idx[r]][0], cmap="gray"); axes[r, 0].axis("off")
    axes[r, 1].axis("off")
    axes[r, 1].text(0, 1, "REAL: " + textwrap.fill(refs[r], 105) + "\n\nGENERATED: " + textwrap.fill(hyps[r], 105),
                    va="top", fontsize=8.5, family="monospace")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/examples.png", dpi=110); plt.show()
print("Files saved in", OUT_DIR, "->", sorted(f for f in os.listdir(OUT_DIR) if not f.startswith("imgcache")))

## Notes
- **Outputs** are in `/kaggle/working/xray_report/`: `best.pt` (weights), `test_predictions.csv`, `test_metrics.json`, `learning_curves.png`, `examples.png`. Use *Save Version → Save & Run All* to keep them and to let the run continue with the browser closed.
- **Image cache:** the first run reads all images once (a few minutes) and stores a small copy in the output folder; later runs reuse it. If you change `IMG_SIZE` a new cache is built.
- **Out of memory:** handled automatically by the pre-flight check (it halves `BATCH_SIZE`). You can also use `ENCODER = "resnet18"`.
- **Quick dry run:** set `MAX_TRAIN = 800`, `EPOCHS = 2`, `N_EVAL = 64`.
- Metrics are computed with plain-Python BLEU/ROUGE (no extra installs). Absolute scores for chest X-ray reports are low by nature; compare runs with each other, and look at the "image check" line to see whether the model really looks at the X-ray.